[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaronJenish/zawdwadw/blob/main/Experiment_17_RNN_Gradient_Analysis_and_Gradient_Clipping.ipynb)


# Experiment 17: RNN Gradient Analysis and Gradient Clipping

CSA47 – Deep Learning. Run the setup cell first, then execute the experiment cells in order.


In [ ]:
# Common setup – run this cell first
import os, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## Experiment 17 – RNN Gradient Analysis and Gradient Clipping

In [ ]:
import tensorflow as tf
# Character-level next-character prediction on a short text corpus
text_data=("deep learning models learn patterns from data. " * 300)
chars=sorted(set(text_data)); c2i={c:i for i,c in enumerate(chars)}
encoded=np.array([c2i[c] for c in text_data])
seq_len=25
Xc=np.array([encoded[i:i+seq_len] for i in range(len(encoded)-seq_len)])
yc=encoded[seq_len:]; vocab=len(chars)
ds=tf.data.Dataset.from_tensor_slices((Xc[:5000],yc[:5000])).batch(64)
def train_clip(clip):
    model=tf.keras.Sequential([tf.keras.layers.Embedding(vocab,32),tf.keras.layers.SimpleRNN(64),tf.keras.layers.Dense(vocab)])
    opt=tf.keras.optimizers.Adam(learning_rate=.03,clipnorm=clip) if clip=="norm" else tf.keras.optimizers.Adam(learning_rate=.03,clipvalue=1.0 if clip=="value" else 1e6)
    loss_fn=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True); norms=[]; losses=[]
    for epoch in range(3):
        for xb,yb in ds:
            with tf.GradientTape() as tape:
                logits=model(xb,training=True); loss=loss_fn(yb,logits)
            grads=tape.gradient(loss,model.trainable_variables)
            norms.append(float(tf.linalg.global_norm(grads))); opt.apply_gradients(zip(grads,model.trainable_variables)); losses.append(float(loss))
    return norms,losses
results={}
for c in ["none","norm","value"]: results[c]=train_clip(c)
for c,(g,l) in results.items(): plt.plot(g,label=c)
plt.yscale("log"); plt.legend(); plt.title("Gradient norm"); plt.show()
for c,(g,l) in results.items(): plt.plot(l,label=c)
plt.legend(); plt.title("Training loss"); plt.show()